# Grundlæggende Data Science 2026 - Exam Project
Niels Abildskov, Magnus Høj, Andreas Arnfred Nielsen

## Part 1. Data Processing
We will be using the ```pandas``` library to represent our data.

## 1.1. Cleaning
To get an initial overview of the data, we will load and preprocess a 250-row sample.

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

tqdm.pandas()
np.random.seed(42) # Seeding for reproducibility

df = pd.read_csv("data/news_sample.csv", index_col=0)
df.head(3)

,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary
0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,Sometimes the power of Christmas will make you...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Church Congregation Brings Gift to Waitresses ...,Ruth Harris,NaN,[''],NaN,NaN,NaN
1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,Zurich Times,NaN,[''],NaN,NaN,NaN
2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,Never Hike Alone: A Friday the 13th Fan Film U...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Never Hike Alone - A Friday the 13th Fan Film ...,NaN,NaN,[''],Never Hike Alone: A Friday the 13th Fan Film ...,NaN,NaN


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 249
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                250 non-null    int64  
 1   domain            250 non-null    object 
 2   type              238 non-null    object 
 3   url               250 non-null    object 
 4   content           250 non-null    object 
 5   scraped_at        250 non-null    object 
 6   inserted_at       250 non-null    object 
 7   updated_at        250 non-null    object 
 8   title             250 non-null    object 
 9   authors           170 non-null    object 
 10  keywords          0 non-null      float64
 11  meta_keywords     250 non-null    object 
 12  meta_description  54 non-null     object 
 13  tags              27 non-null     object 
 14  summary           0 non-null      float64
dtypes: float64(2), int64(1), object(12)
memory usage: 31.2+ KB


In [3]:
print(f"Percentage of type rows null: {df["type"].notna().sum() / len(df["type"])*100:.2f}%")

Percentage of type rows null: 95.20%


From this we can already make the following observations about the different columns:
2. Id: We need to verify this, but this is probably a unique identifier for the row.
3. Domain: The domain from which the article is scraped.
4. Type: The actual label for the reliability of the article.
5. Url: The link to the article.
6. Content: The actual content of the article.
7. scraped_at, inserted_at, updated_at: Timestamps recording certain alterations of the data.
8. Title: The title of the article.
9. Authors: The list of authors for the article
10. Keywords: All null.
11. Meta-keywords: We need to analyze this column further to distinguish it from keywords.
12. Meta-description: Description of the article.
13. Tags: Probably the same as keywords.
14. Summary: All null

Now we have the following tasks:
1. Drop uninformative columns. These are:
    - scraped_at
    - inserted_at
    - updated_ad
    - authors: We remove authors since a name itself doesn't imply anything about the content's reliability. The only thing a model would learn would be that certain authors are less reliable than others.
    - keywords, tags and meta_keywords: It seems that only a subset of rows have either of these, which will risk biasing modelling.
    - meta_description and summary: Same argument as above.
2. Deduplicate content. For modelling, we do not want duplicates since they will bias the model.
3. Remove rows where either type or content is missing. If it only a handful of our data points was missing a type, we could probably manually impute the type value, however given that ~5% of our data has a missing value in the type column, we cannot reliably impute the values, and for 1M rows, this would mean we have ~50.000 rows of missing values.
4. Cast the non-null type values into a ```pandas``` category type.

**Note:** Given that each column has a unique identifier (```Id```), we can safely drop columns, since we can always get the data from the raw dataset.

In [4]:
from preprocessing.cleaning import filter_dataset
filtered_df = filter_dataset(
    df,
    drop_cols=["scraped_at", "inserted_at", "updated_at", "authors", "keywords", "tags", "meta_keywords", "meta_description", "summary"],
    remove_nulls_cols=["type", "content"],
    deduplicate_cols=["content"],
    convert_to_cat_cols=["type"])
filtered_df.info()

Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.


<class 'pandas.core.frame.DataFrame'>
Index: 227 entries, 0 to 246
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   id       227 non-null    int64   
 1   domain   227 non-null    object  
 2   type     227 non-null    category
 3   url      227 non-null    object  
 4   content  227 non-null    object  
 5   title    227 non-null    object  
dtypes: category(1), int64(1), object(4)
memory usage: 11.2+ KB


Now that our dataset is in some kind of shape for further processing, we can begin cleaning it more thoroughly. Given that we're working with natural language, we can expect to encounter multiple types of elements that are not directly relevant to the meaning of the given text. This includes:
1. Non-NLP entities: We will be performing masking on e-mails, dates, URl's and numbers, however this is not exhaustive. Other entities could include adresses, names, locations etc..
2. Punctuation: It is debatable whether punctuation directly contributes to a document's semantic meaning, but for initial cleaning purposes, we will keep it.
3. Casing: In most cases, we can expect words to have the same meaning regardless of the casing of their characters. There are exceptions though, but due to limited compute resources, we will favour a smaller vocabulary over 100% precision.

We will be cleaning the texts using our own date-matching regular expression and the ```clean-text``` library. Additionally, the only special characters we will remove are ```<``` and ```>``` since these will be used for our masking tokens.

Let's sample a document before and after cleaning:

In [5]:
sample = filtered_df.iloc[1].content
print(sample)

AWAKENING OF 12 STRANDS of DNA – “Reconnecting with You” Movie

% of readers think this story is Fact. Add your two cents.

Headline: Bitcoin & Blockchain Searches Exceed Trump! Blockchain Stocks Are Next!

[January 24, 2018 - ZurichTimes.net]

As Miles Johnston was giving update, it was another case of Strange Synchronicities of Goodness Hidden inside of Tests and Trials, like a Follow the WhiteRabbit down the Rabbit Hole type of exercise.

In Researching the 12 Strands of DNA we came across some articles, one in particular was as a Strange Synchronicity written exactly 1 year ago on the Same Topic.

https://www.youtube.com/watch?v=_6Ze1mrs4BQ

https://www.youtube.com/watch?v=2nKcDiIc8JY

What are the 12 Strands of our DNA and Why is a War Against our DNA?

Trailer for Awakening of 12 Strands

The Full Video is only available as a Paid Video on Vimeo.

AWAKENING OF 12 STRANDS – “Reconnecting with You”

vimeo.com/ondemand/Awakeningof12strands

AWAKENING OF 12 STRANDS – “Reconnecting wi

In [6]:
from preprocessing.cleaning import clean_text, CleaningConfig

clean_text(sample, CleaningConfig())

'awakening of <NUM> strands of dna "reconnecting with you" movie % of readers think this story is fact. add your two cents. headline: bitcoin & blockchain searches exceed trump! blockchain stocks are next! [january <NUM>, <NUM> - zurichtimes.net] as miles johnston was giving update, it was another case of strange synchronicities of goodness hidden inside of tests and trials, like a follow the whiterabbit down the rabbit hole type of exercise. in researching the <NUM> strands of dna we came across some articles, one in particular was as a strange synchronicity written exactly <NUM> year ago on the same topic. <URL> <URL> what are the <NUM> strands of our dna and why is a war against our dna? trailer for awakening of <NUM> strands the full video is only available as a paid video on vimeo. awakening of <NUM> strands "reconnecting with you" vimeo.com/ondemand/awakeningof<NUM>strands awakening of <NUM> strands "reconnecting with you". from sandra daroy on vimeo. we have not watched the full

In [7]:
from preprocessing.cleaning import clean_data

cleaned = clean_data(filtered_df, CleaningConfig())

In [8]:
cleaned.head()

,id,domain,type,url,content,title
0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,sometimes the power of christmas will make you...,church congregation brings gift to waitresses ...
1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,awakening of <NUM> strands of dna – “reconnect...,awakening of <NUM> strands of dna – “reconnect...
2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,never hike alone: a friday the <NUM>th fan fil...,never hike alone - a friday the <NUM>th fan fi...
3,768,awm.com,unreliable,http://awm.com/elusive-alien-of-the-sea-caught...,"when a rare shark was caught, scientists were ...",elusive ‘alien of the sea ‘ caught by scientis...
4,791,bipartisanreport.com,clickbait,http://bipartisanreport.com/2018/01/21/trumps-...,donald trump has the unnerving ability to abil...,trump’s genius poll is complete & the results ...


## 1.2. Tokenization
For the tokenization step, we remove all special characters but ```<>``` and split on whitespace. Then we perform stopwords removal as well as simple stemming.

To argue for the two latter methods's effectiveness, we will now be demonstrating vocabulary shrinkage when they are employed.

In [ ]:
from preprocessing.tokenization import TokenizerConfig , Tokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
import nltk

nltk.download('stopwords')


cfg = TokenizerConfig(
    top_n=None, 
    special_tokens=[],
    stopwords=[],
    stemmer=None)
tk = Tokenizer(cfg, mutable=True)
tk.fit_tokenizer(cleaned["content"])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nva\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 227/227 [00:00<00:00, 4705.25it/s]


16286

In [14]:
cfg = TokenizerConfig(
    top_n=None, 
    special_tokens=["<DATE>", "<NUM>", "<URL>", "<EMAIL>"],
    stopwords=stopwords.words("english"),
    stemmer=None)
tk = Tokenizer(cfg, mutable=True)
tk.fit_tokenizer(cleaned["content"])

100%|██████████| 227/227 [00:00<00:00, 1479.47it/s]


16154

In [15]:
cfg = TokenizerConfig(
    top_n=None, 
    special_tokens=["<DATE>", "<NUM>", "<URL>", "<EMAIL>"],
    stopwords=stopwords.words("english"),
    stemmer=PorterStemmer())
tk = Tokenizer(cfg, mutable=True)
tk.fit_tokenizer(cleaned["content"])

100%|██████████| 227/227 [00:00<00:00, 367.38it/s]


10850

In [ ]:
cfg = TokenizerConfig(
    top_n=None, 
    special_tokens=["<DATE>", "<NUM>", "<URL>", "<EMAIL>"],
    stopwords=stopwords.words("english"),
    stemmer=PorterStemmer())
tk = Tokenizer(cfg, mutable=True)
tk.fit_tokenizer(cleaned["content"])

# 2. Expanding the dataset

In [9]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

full_dataset = "data/995,000_rows.csv"
cleaned_ds_path = "data/995k_rows_cleaned.csv"
chunk_size = 2048

# We need row-count to get tqdm to work, so a hacky solution is to just read the id-column and count the rows
row_count = len(pd.read_csv(full_dataset, usecols=["id"]))
print(f"Reading in dataset with {row_count} rows.")
large_df = pd.read_csv(
    full_dataset, 
    index_col=0, 
    usecols=["id", "domain", "type", "url", "content", "title"],
    chunksize=chunk_size)
large_df

Reading in dataset with 995000 rows.


C:\Users\nva\AppData\Local\Temp\ipykernel_35796\750843502.py:11: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  row_count = len(pd.read_csv(full_dataset, usecols=["id"]))


In [10]:
first_chunk = True

chunk_count = (row_count // chunk_size) + 1
print(f"Cleaning dataset in {chunk_count} chunks\n")
for chunk in tqdm(large_df, total=chunk_count):
    filtered = filter_dataset(
        chunk,
        drop_cols=[], # didn't even load them this time
        remove_nulls_cols=["type", "content"],
        deduplicate_cols=["content"],
        convert_to_cat_cols=["type"])
    cleaned = clean_data(filtered, CleaningConfig())

    cleaned.to_csv(
        cleaned_ds_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False)
    first_chunk = False

Cleaning dataset in 486 chunks



100%|██████████| 486/486 [01:30<00:00,  5.37it/s]


In [11]:
new_df = pd.read_csv(cleaned_ds_path)
new_df.head()

,domain,type,url,content,title
0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,plus one article on google plus (thanks to ali...,iran news round up
1,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,the cost of the best senate banking committee ...,the cost of the best senate banking committee ...
2,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,man awoken from <NUM>-year coma commits suicid...,man awoken from <NUM>-year coma commits suicid...
3,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,when julia geist was asked to draw a picture o...,opening a gateway for girls to enter the compu...
4,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,– <NUM> compiled studies on vaccine dangers (a...,<NUM> compiled studies on vaccine dangers – in...


In [12]:
len(new_df)

832651

# Three points about the data
1. Check how large a fraction of the entire vocabulary is represented in an average document
2. Check the type distribution across sources
3. Type distribution in general
4. Zipf's law.
5. Article lengths vs. "fakeness" score (any correlation?)